<a href="https://colab.research.google.com/github/julianbarrycodes/SmartTicketIssuer/blob/master/fetch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Step 1 of the clinical trials landscape project:
fetch a small sample of trials from the ClinicalTrials.gov API (v2)
and look at what comes back.

Setup (once):
    pip install requests pandas

Run:
    python fetch_trials.py
"""

import requests
import pandas as pd

API_URL = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.cond": "diabetes",  # try other conditions later, e.g. "obesity", "Alzheimer"
    "pageSize": 100,           # max 1000 per page
    "format": "json",
}

response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()
studies = response.json().get("studies", [])

rows = []
for study in studies:
    ps = study.get("protocolSection", {})
    ident = ps.get("identificationModule", {})
    status = ps.get("statusModule", {})
    design = ps.get("designModule", {})
    sponsor = ps.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})

    rows.append({
        "nct_id": ident.get("nctId"),
        "title": ident.get("briefTitle"),
        "status": status.get("overallStatus"),
        "phases": ", ".join(design.get("phases", [])),
        "study_type": design.get("studyType"),
        "sponsor": sponsor.get("name"),
        "sponsor_type": sponsor.get("class"),
        "start_date": status.get("startDateStruct", {}).get("date"),
    })

df = pd.DataFrame(rows)

print(f"Fetched {len(df)} trials\n")
print(df.head(10).to_string())
print("\nTrials by status:")
print(df["status"].value_counts())
print("\nTrials by phase:")
print(df["phases"].value_counts())

# Save so you can explore it later (Excel, Power BI, or more Python)
df.to_csv("trials_sample.csv", index=False)
print("\nSaved to trials_sample.csv")

Fetched 100 trials

        nct_id                                                                                                                  title      status  phases      study_type                                       sponsor sponsor_type  start_date
0  NCT03162237                                                                Safety and Efficacy Study of Islets Xenotransplantation   COMPLETED      NA  INTERVENTIONAL                                   Wei Wang,MD        OTHER     2013-07
1  NCT01121172                                                          Apelin Levels and G212A Polymorphism of Apelin Receptor (APJ)   COMPLETED           OBSERVATIONAL          Aristotle University Of Thessaloniki        OTHER     2010-05
2  NCT01630512  Mindfulness-Based Cognitive Therapy (MBCT) and Cognitive Behavioral Therapy (CBT) for Depression in Diabetes Patients   COMPLETED      NA  INTERVENTIONAL           University Medical Center Groningen        OTHER     2011-05
3  NCT01698112  